# Lab 1 — Semantic Search: How Vector Search Actually Works

**Thesis:** You can run state-of-the-art semantic search — finding documents by *meaning*, not just matching keywords — without writing a single line of embedding code. Elasticsearch handles it server-side through the Elastic Inference Service.

## What you'll learn
- How `semantic_text` fields work and why you write zero embedding code
- The 4-step query mechanism: ES → Elastic Inference Service (EIS) → Jina v5 → ANN
- Why semantic search finds concepts your users never explicitly typed
- How ES chunks and stores document embeddings, and why that matters

## Before you start
This notebook reads credentials from environment variables:  
- **In Instruqt:** `ES_ENDPOINT` and `ES_API_KEY` are pre-configured — just run the cells.  
- **Re-running from the repo:** `export ES_ENDPOINT=https://...` and `export ES_API_KEY=...`

In [ ]:
# --- Workshop helpers ---
# Defined inline so this notebook is self-contained and runs from the repo too.

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=60)

def show_hits(resp, fields=("id", "title", "trap_type"), score=True):
    """Pretty-print search hits as a ranked table."""
    hits = resp["hits"]["hits"]
    if not hits:
        print("  (no hits)"); return
    for rank, h in enumerate(hits, 1):
        src = h.get("_source", {})
        cols = "  ".join(str(src.get(f, "")) for f in fields)
        s = f"  {h['_score']:.4f}" if score and h.get("_score") is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

# Retriever builders — same query string, different strategy
def r_semantic(query):
    return {"standard": {"query": {"semantic": {"field": "body_semantic", "query": query}}}}

def r_bm25(query):
    return {"standard": {"query": {"multi_match": {
        "query": query, "fields": ["title^3", "body"], "type": "best_fields"}}}}

def r_rrf(query, rank_constant=60, rank_window_size=100):
    return {"rrf": {"retrievers": [r_bm25(query), r_semantic(query)],
                    "rank_constant": rank_constant, "rank_window_size": rank_window_size}}

def search(retriever, size=5, source=("id","title","trap_type","version_tags")):
    return es.search(index=INDEX, retriever=retriever, size=size, source=list(source))

print("✓ Helpers loaded")

In [ ]:
# Sanity check: confirm we're connected and the corpus is indexed.
info = es.info()
print(f"Connected to Elasticsearch {info['version']['number']}")

count = es.count(index=INDEX)["count"]
print(f"Index '{INDEX}': {count} documents indexed")

if count == 0:
    print("\n⚠ No documents found — ingest may still be running. Wait 30s and retry.")

## What does a `semantic_text` field look like in practice?

The corpus has a field called `body_semantic` that was mapped as `semantic_text`. Let's look at the actual mapping to understand what Elasticsearch stores.

Pay attention to:
- `type: semantic_text` — the field type that enables all of this
- `inference_id` — notice it's **auto-assigned** (`.jina-embeddings-v5-text-small`). You didn't set it; Elastic Serverless provisioned it for you.
- The nested `.embedding` sub-field — that's where the vectors live (we'll inspect it in a moment)

In [ ]:
# Fetch the index mapping and focus on the semantic_text field.
mapping = es.indices.get_mapping(index=INDEX)
props = mapping[INDEX]["mappings"]["properties"]

print("=== body_semantic field mapping ===")
print(json.dumps(props.get("body_semantic", {}), indent=2))

print("\n=== all field types ===")
for field, defn in props.items():
    ftype = defn.get("type", "object")
    print(f"  {field:<20} {ftype}")

## The model behind `semantic_text` — and why it matters

That `.jina-embeddings-v5-text-small` endpoint was **automatically provisioned** when we created the index. You never had to sign up for an external embedding API, manage credentials, or write vectorization code.

The **Elastic Inference Service (EIS)** is a managed embedding + reranking + LLM gateway built into the platform. At index time: ES sends the document body to EIS → EIS calls the model → the resulting vector is stored alongside the text. Same thing at query time — your query string gets embedded by the same model automatically.

### Why Jina v5?

A few things make it well-suited for retrieval specifically:

- **32,768-token context window** — most embedding models cap at 512–8,192 tokens. Jina v5 can hold an entire long document in context before splitting it into chunks. That context window is why late chunking (next cell) works.
- **Task-specific adapters** — Jina v5 uses different adapter weights for retrieval vs. classification vs. clustering. For search queries, the `retrieval` adapter is active, which is tuned for **asymmetric matching**: short query against long document passage.
- **Matryoshka Representation Learning (MRL)** — meaning is packed front-loaded into the early dimensions of each vector. This means you can truncate a 1024-dim vector to 256 dims and retain ~95% of the retrieval quality. Useful for cost/storage optimization at scale.
- **Published benchmarks** — the model's performance is reproducible and verifiable on MTEB (Massive Text Embedding Benchmark), not just a marketing claim.

All of this runs server-side. You interact with it through one field mapping.

Let's inspect the live endpoint:


In [ ]:
# Show all active inference endpoints on this project.
# We're looking for the .jina-embeddings-v5-text-small endpoint.
endpoints = es.inference.get()

for ep in endpoints.get("endpoints", []):
    ep_id = ep.get("inference_id", "")
    service = ep.get("service", "")
    task = ep.get("task_type", "")
    print(f"  {ep_id}")
    print(f"    service={service}  task={task}")
    print()

## Your first semantic query — note: zero embedding code

Here's the query we're about to run:

```
"securing cluster traffic"
```

The top document is about **TLS / transport-layer encryption**. The document body doesn't contain the phrase "securing cluster traffic" — but it's *semantically about* securing cluster traffic.

We're just calling `r_semantic(query)` and printing results. No client-side embedding. No vector math. No numpy. The `semantic` query type in the retriever DSL handles it all server-side.

In [ ]:
query = "securing cluster traffic"
print(f"Query: {query!r}\n")

resp = search(r_semantic(query))
show_hits(resp)

# Print the top hit's body so we can verify it's semantically relevant
top = resp["hits"]["hits"][0]["_source"]
print(f"\nTop hit body preview:")
print(f"  {top.get('body','')[:300]}...")

## What just happened? The 4-step mechanism

```
Your query string
      │
      ▼
  Elasticsearch  ──── sends query text ───►  Elastic Inference Service (EIS)
      │                                              │
      │                                    Jina v5 embedding model
      │                                              │
      │          ◄── 1024-dim float vector ──────────┘
      │
      ▼
  HNSW ANN index  (approximate nearest-neighbour search over stored doc vectors)
      │
      ▼
  Top-K semantically similar documents
```

**ANN / HNSW** — Elasticsearch uses Hierarchical Navigable Small World graphs for approximate nearest-neighbour search. "Approximate" means it trades a tiny accuracy margin for being **orders of magnitude faster** than brute-force cosine similarity over millions of vectors. At 60 docs it's exact; at 10M it's still fast.

**Why this matters:** at index time, when you wrote `body_semantic: doc["body"]`, ES sent the text to EIS, got a vector back, and stored it in the HNSW index — no separate pipeline, no Spark job, no ML framework. At query time, the same thing happens to your query string, and the ANN search runs. You wrote none of this.

In [ ]:
# More wow queries — these all find semantically related docs without keyword overlap.

wow_queries = [
    "how do I back up my cluster data",
    "users can't connect to Kibana",
]

for q in wow_queries:
    print(f"\n{'='*60}")
    print(f"QUERY: {q!r}")
    resp = search(r_semantic(q))
    show_hits(resp)
    # Show what the top result is actually about
    if resp["hits"]["hits"]:
        top = resp["hits"]["hits"][0]["_source"]
        print(f"  → top hit: {top.get('title','')}")

## How chunking works — and what `semantic_text` hides from you

Even with a 32K token context window, you don't always want to embed an entire document as one vector. Here's why: ANN search ranks *chunks*, not whole documents. If a 10,000-word document has one relevant paragraph buried in section 7, you want that paragraph to surface — not the entire document.

**The default `semantic_text` chunking strategy (sentence-based):**
- Target chunk size: ~250 words
- Overlap: 1 sentence between adjacent chunks
- Boundary detection: respects sentence endings, not arbitrary character counts

**Why not always use the maximum context window?**
Bigger chunks = more context per embedding, but also more noise. The relevant signal gets diluted. Smaller chunks = sharper retrieval precision, but you lose context for cross-sentence references. The 250-word default is a practical balance. You can tune `chunking_settings` on the `semantic_text` field if your corpus has different characteristics — but the default works well for documentation-style content.

**What happens at query time?** Your query vector is compared against *all* chunks across *all* documents. The **max similarity** across any chunk becomes the document's score. So a 5-chunk document competes fairly with a 1-chunk document.

Let's look inside a stored document to see the chunks:


In [ ]:
# Fetch doc-010 directly and inspect its stored semantic chunks.
# doc-010 is the TLS cluster communications doc — a good length to inspect.
doc = es.get(index=INDEX, id="doc-010")
src = doc["_source"]

print(f"Document: {src.get('id')} — {src.get('title')}")
print(f"Body length: {len(src.get('body',''))} chars")

# The embedding data lives in body_semantic
sem = src.get("body_semantic", {})
if isinstance(sem, dict):
    chunks = sem.get("chunks", [])
    print(f"\nStored as {len(chunks)} semantic chunk(s):")
    for i, chunk in enumerate(chunks):
        text = chunk.get("text", "")
        emb  = chunk.get("embeddings", [])
        dims = len(emb) if isinstance(emb, list) else "?"
        print(f"  Chunk {i+1}: {len(text)} chars → {dims}-dim vector")
        print(f"    Preview: {text[:120].strip()}...")
else:
    # Serverless may return the raw text at the top level (model-encoded)
    print(f"body_semantic (raw): {str(sem)[:200]}")

## Late chunking — context-aware embeddings

Standard chunking has a problem: each chunk is embedded **without context from the rest of the document**. Consider a doc where paragraph 1 introduces "the cluster" and paragraph 4 says "it crashed." Chunked separately, paragraph 4's embedding doesn't know what "it" refers to. The embedding is generic.

**Jina v5's late chunking** solves this:
1. Run the *entire document* through the model (up to 32K tokens) to produce token-level vectors with full document context
2. *Then* pool those token vectors into chunk boundaries

The result: the chunk covering "it crashed" retains the semantic context that "it" = the cluster from earlier in the document.

### In practice: inspect the endpoint config

Late chunking is a toggle on the inference endpoint. Let's check if it's enabled:


In [ ]:
# Check if late chunking is enabled on the Jina v5 endpoint.
# Late chunking embeds the full document context first, THEN splits into chunks —
# so cross-paragraph references stay semantically coherent.
endpoints = es.inference.get()
for ep_type, eps in endpoints.body.items():
    for ep in (eps if isinstance(eps, list) else [eps]):
        if "jina" in str(ep.get("inference_id","")).lower():
            task_settings = ep.get("task_settings", {})
            service_settings = ep.get("service_settings", {})
            print(f"Endpoint: {ep.get('inference_id')}")
            print(f"  task_settings:    {task_settings}")
            print(f"  service_settings: {service_settings}")
            print()
            late = task_settings.get("late_chunking", service_settings.get("late_chunking", "not set"))
            print(f"  Late chunking: {late}")


## Matryoshka dimensions — storage vs. recall trade-off

Jina v5 uses Matryoshka Representation Learning (MRL): the model is trained so that the **first N dimensions of every vector already capture most of the semantic meaning**. Later dimensions add precision, but at rapidly diminishing returns.

This means you can truncate stored vectors to save space with minimal retrieval loss:

| Dimensions | Storage vs. full | Approximate recall retention |
|---|---|---|
| 1024 (default) | 100% | 100% |
| 512 | 50% | ~99% |
| 256 | 25% | ~95% |
| 128 | 12.5% | degrades noticeably |

**In practice:** For most RAG use cases, 1024 dims (the default) is fine. At very large scale (hundreds of millions of documents), 512 or 256 dims can cut infrastructure costs significantly with acceptable recall loss. This is configured in the mapping when you create the index — not something you can change post-index without reindexing.

**Also relevant: BBQ (Binary Quantization)**  
At extreme scale (billions of vectors), Elasticsearch supports BBQ — compressing each dimension to 1 bit. 96% storage reduction, ~95% recall. Requires specific Elasticsearch versions. Mentioned here as a "when you hit scale" tool, not something you'll configure today.


## Summary — and a question

You just ran semantic search that:
- Finds documents by **concept**, not keyword overlap
- Uses a **production-grade embedding model** (Jina v5, 1024 dims) managed by Elastic
- Handles **chunking, embedding, and ANN indexing** automatically
- Requires **zero ML infrastructure** from you — just `semantic_text` in the mapping

---

**The setup question for Lab 2:**

> If semantic search is this good at understanding meaning, why would you ever need old-fashioned keyword (BM25) search?

Try this in the **Dev Console** (or here):

```python
show_hits(search(r_semantic("exit code 137")))
```

Does the result look right? What doc should be #1 for that query? Lab 2 will show you exactly why semantic search fails here — and why you need both.

---
*Continue in Dev Console → Lab 2 assignment, or open `lab2-where-vector-breaks.ipynb`*